# Polymarket — Closed sports-market dataset

## Goal

Create a dataset of **closed Polymarket sports-market candidates**.

For this stage, we do **not**:

- determine the resolved outcome;
- manually decide whether a market is an actual sports match;
- fetch historical CLOB prices;
- restrict to binary markets.

Those steps come later.

### Dataset definition

Instead of downloading every closed Polymarket market and trying to identify sports locally, we query Gamma directly using Polymarket's own:

- **Sports tag** (`tag_id = 1`)
- `related_tags = true`

Gamma therefore expands the Sports tag server-side to markets associated with related sports tags.

This gives us a much smaller and more defensible candidate universe for later manual annotation.


In [ ]:
import time
from pathlib import Path

import pandas as pd
import requests

GAMMA_BASE_URL = "https://gamma-api.polymarket.com"

# Polymarket's official Sports tag
SPORTS_TAG_ID = 1

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


## 1. Verify the Sports tag

Before collecting markets, verify what `SPORTS_TAG_ID` refers to.

This is mainly documentation / a sanity check. We do **not** manually define all related sports tags in Python; Gamma handles that expansion when we later pass `related_tags=true`.


In [ ]:
response = requests.get(
    f"{GAMMA_BASE_URL}/tags/{SPORTS_TAG_ID}",
    timeout=30,
)

response.raise_for_status()

sports_tag = response.json()
sports_tag

In [ ]:
print("Tag ID:", sports_tag.get("id"))
print("Label:", sports_tag.get("label"))
print("Slug:", sports_tag.get("slug"))


In [ ]:
response = requests.get(
    f"{GAMMA_BASE_URL}/tags/{SPORTS_TAG_ID}/related-tags/tags"
)

response.raise_for_status()

related_sports_tags = response.json()
related_sports_tags

## 2. Fetch closed sports markets

We use Gamma's **keyset pagination** endpoint because ordinary offset pagination does not scale to large result sets.

The important filters are:

```python
closed=true
tag_id=SPORTS_TAG_ID
related_tags=true
```

So we request only closed markets connected to Polymarket's Sports taxonomy.

`next_cursor` from each response is passed back as `after_cursor` for the next page.


In [ ]:
def fetch_closed_sports_markets(
    page_size=100,
    max_pages=None,
    sleep_seconds=0.1,
):
    all_markets = []
    after_cursor = None
    page = 0

    while True:
        params = {
            "closed": "true",
            "tag_id": SPORTS_TAG_ID,
            "related_tags": "true",
            "limit": page_size,
        }

        if after_cursor is not None:
            params["after_cursor"] = after_cursor

        response = requests.get(
            f"{GAMMA_BASE_URL}/markets/keyset",
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        payload = response.json()

        batch = payload.get("markets", [])
        next_cursor = payload.get("next_cursor")

        if not batch:
            break

        all_markets.extend(batch)

        print(
            f"Page {page + 1}: {len(batch):>3} markets "
            f"| total sports candidates = {len(all_markets)}"
        )

        page += 1

        if max_pages is not None and page >= max_pages:
            break

        if not next_cursor:
            break

        after_cursor = next_cursor
        time.sleep(sleep_seconds)

    return all_markets


### Run the collection

For the final dataset, `max_pages=None` is fine here because the filtering happens **inside Gamma**, before the data are returned.

If you want a quick test first, temporarily use something like `max_pages=3`.


In [ ]:
raw_markets = fetch_closed_sports_markets(
    page_size=100,
    max_pages=None,
)

markets_df = pd.DataFrame(raw_markets)

print("\nTotal closed sports candidates:", len(markets_df))
print("Columns:", len(markets_df.columns))

markets_df.head()


## 3. Inspect what Polymarket's "sports" returned

We inspect the returned markets to understand:

- how many markets were collected;
- which original `category` values appear;
- whether "miscategorized" sports markets are recovered.

A market can therefore appear in this dataset even if its old `category` field is something other than `Sports`, because the query is based on Polymarket's **tag taxonomy**, not solely the category column.


In [ ]:
print("Total rows:", len(markets_df))

if "category" in markets_df.columns:
    print("\nOriginal category values:")
    display(
        markets_df["category"]
        .value_counts(dropna=False)
        .rename_axis("category")
        .reset_index(name="count")
    )


### Check the Khabib example (he was misclassified as pop-culture category before)

> **Will Khabib win his UFC 254 Fight?**

Serves as a sanity check for whether the Sports-tag query catches sports markets that the old category field missed.


In [ ]:
khabib_check = markets_df[
    markets_df["question"].str.contains(
        "Khabib",
        case=False,
        na=False,
    )
]

display_columns = [
    col for col in [
        "id",
        "question",
        "category",
        "endDate",
    ]
    if col in khabib_check.columns
]

khabib_check[display_columns]


## 4. Build the dataset

At this stage we preserve the candidate sports markets rather than making further analytical decisions.

In particular:

- a row does **not** need to represent a head-to-head match yet;
- a row does **not** need to be binary yet;
- `outcomePrices` are retained as raw Polymarket metadata, not interpreted as labels;
- CLOB token IDs are retained for later historical-price work.

Stable identifiers such as `market_id` and `condition_id` will let later stages join annotations, resolved outcomes, and price histories back onto these rows.


In [ ]:
sports_df = markets_df.copy()

# Convert useful time fields when present
for col in ["startDate", "endDate", "createdAt", "closedTime"]:
    if col in sports_df.columns:
        sports_df[col] = pd.to_datetime(
            sports_df[col],
            utc=True,
            errors="coerce",
        )

# Convert useful numeric metadata when present
for col in ["volumeNum", "liquidityNum"]:
    if col in sports_df.columns:
        sports_df[col] = pd.to_numeric(
            sports_df[col],
            errors="coerce",
        )


wanted_columns = [
    # Stable identifiers
    "id",
    "conditionId",
    "questionID",

    # Market text
    "question",
    "slug",
    "description",

    # Timing / status
    "startDate",
    "endDate",
    "createdAt",
    "closedTime",
    "closed",

    # Original Polymarket metadata
    "category",
    "subcategory",
    "tags",
    "events",

    # Outcomes / token metadata for later stages
    "outcomes",
    "outcomePrices",
    "clobTokenIds",

    # Market metadata
    "volumeNum",
    "liquidityNum",
    "resolutionSource",
]

available_columns = [
    col for col in wanted_columns
    if col in sports_df.columns
]

closed_sports_markets = sports_df[available_columns].copy()

closed_sports_markets = closed_sports_markets.rename(
    columns={
        "id": "market_id",
        "conditionId": "condition_id",
        "questionID": "question_id",
    }
)

closed_sports_markets.head()


## 5. Basic dataset checks

These checks are only about collection quality. We are not yet deciding whether a market is a suitable match-level observation for the eventual analysis.


In [ ]:
print("Rows:", len(closed_sports_markets))

if "market_id" in closed_sports_markets.columns:
    print(
        "Duplicate market IDs:",
        closed_sports_markets["market_id"].duplicated().sum(),
    )

if "closed" in closed_sports_markets.columns:
    print(
        "Rows not marked closed:",
        (~closed_sports_markets["closed"].fillna(False)).sum(),
    )

if "category" in closed_sports_markets.columns:
    print("\nOriginal categories:")
    print(
        closed_sports_markets["category"]
        .value_counts(dropna=False)
        .head(20)
    )


In [ ]:
preview_columns = [
    col for col in [
        "market_id",
        "question",
        "category",
        "endDate",
        "outcomes",
    ]
    if col in closed_sports_markets.columns
]

closed_sports_markets[preview_columns].head(50)


## 6. Save the Task 1 dataset

This notebook produces one dataset:

```text
data/closed_sports_markets.csv
```

The next project stages can then work from this fixed candidate universe:

```text
closed sports candidates
        ↓
resolved outcomes / labels
        ↓
manual annotation of actual sports matches
        ↓
historical prices
        ↓
analysis
```


In [ ]:
OUTPUT_PATH = DATA_DIR / "closed_sports_markets.csv"

closed_sports_markets.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Saved {len(closed_sports_markets)} closed sports-market candidates")
print(OUTPUT_PATH)


# Notes on the selection rule

The selection rule for this dataset is intentionally simple and reproducible:

> Include a market if Gamma returns it for  
> `closed=true`, `tag_id=Sports`, `related_tags=true`.

We do **not** supplement that result with our own sports keyword list.

This means Polymarket's tag taxonomy defines the initial candidates. Any later questions such as:

- "Is this really a sports market?"
- "Is this an individual match?"
- "Which sport / league is this?"
- "Should this row be included in the final analysis?"

belongs to the later **manual annotation** stage rather than this collection notebook.
